# Sistema de clasificación de imágenes

In [1]:
import os
import zipfile
import shutil
import random
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPool2D,
    Dense,
    GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


2026-02-16 00:21:40.973643: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-16 00:21:40.974012: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 00:21:41.022134: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 00:21:44.130686: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

## Descomprimir los Gatos y Perros

In [3]:
import zipfile
from pathlib import Path

zip_path = Path("../data/raw/dogs-vs-cats.zip")
extract_path = Path("../data/raw")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Zip descomprimido correctamente")


Zip descomprimido correctamente


In [4]:
print(os.listdir("../data/raw/dogs-vs-cats"))


['.DS_Store', 'sampleSubmission.csv', 'train']


In [5]:
original_train_dir = Path("../data/raw/dogs-vs-cats/train")
base_dir = Path("../data/raw/dogs-vs-cats/dataset_split")

train_dir = base_dir / "train"
val_dir = base_dir / "validation"

# Crear carpetas
for folder in [
    train_dir / "cats",
    train_dir / "dogs",
    val_dir / "cats",
    val_dir / "dogs"
]:
    folder.mkdir(parents=True, exist_ok=True)

# Listar imágenes
images = list(original_train_dir.glob("*.jpg"))

random.seed(42)
random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
val_images = images[split_idx:]

# Copiar imágenes
for img_path in train_images:
    if "cat" in img_path.name:
        shutil.copy(img_path, train_dir / "cats")
    else:
        shutil.copy(img_path, train_dir / "dogs")

for img_path in val_images:
    if "cat" in img_path.name:
        shutil.copy(img_path, val_dir / "cats")
    else:
        shutil.copy(img_path, val_dir / "dogs")

print("Dataset dividido correctamente")


Dataset dividido correctamente


In [6]:
print(os.listdir("../data/raw/dogs-vs-cats/dataset_split"))


['validation', 'train']


## Generadores (versión estable)

para no matar RAM:

> - IMG_SIZE = 128
>
> - BATCH_SIZE = 8

In [7]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 8

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

trdata = train_datagen.flow_from_directory(
    "../data/raw/dogs-vs-cats/dataset_split/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

tsdata = val_datagen.flow_from_directory(
    "../data/raw/dogs-vs-cats/dataset_split/validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)


Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


## Modelo CNN

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, GlobalAveragePooling2D

model = Sequential()

model.add(Conv2D(32, (3,3), padding="same", activation="relu", input_shape=(128,128,3)))
model.add(MaxPool2D((2,2)))

model.add(Conv2D(64, (3,3), padding="same", activation="relu"))
model.add(MaxPool2D((2,2)))

model.add(Conv2D(128, (3,3), padding="same", activation="relu"))
model.add(MaxPool2D((2,2)))

model.add(GlobalAveragePooling2D())

model.add(Dense(128, activation="relu"))
model.add(Dense(2, activation="softmax"))

model.summary()


/home/vscode/.local/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-16 00:35:15.013852: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,018 (429.76 KB)

 Trainable params: 110,018 (429.76 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"])

In [10]:
history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=3)

Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 247s 98ms/step - accuracy: 0.5940 - loss: 0.6582 - val_accuracy: 0.6396 - val_loss: 0.6284
Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 240s 96ms/step - accuracy: 0.6762 - loss: 0.5988 - val_accuracy: 0.6690 - val_loss: 0.6117
Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 232s 93ms/step - accuracy: 0.7185 - loss: 0.5542 - val_accuracy: 0.7224 - val_loss: 0.5320


In [11]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

trdata = train_datagen.flow_from_directory(
    "../data/raw/dogs-vs-cats/dataset_split/train",
    target_size=(128,128),
    batch_size=8,
    class_mode="categorical"
)

tsdata = val_datagen.flow_from_directory(
    "../data/raw/dogs-vs-cats/dataset_split/validation",
    target_size=(128,128),
    batch_size=8,
    class_mode="categorical"
)

Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [12]:

earlystop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


In [13]:
history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=10,
    callbacks=[earlystop]
)


Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 276s 110ms/step - accuracy: 0.7232 - loss: 0.5456 - val_accuracy: 0.7692 - val_loss: 0.4800
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 275s 110ms/step - accuracy: 0.7509 - loss: 0.5045 - val_accuracy: 0.7568 - val_loss: 0.4924
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 275s 110ms/step - accuracy: 0.7800 - loss: 0.4620 - val_accuracy: 0.7476 - val_loss: 0.5470
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 274s 110ms/step - accuracy: 0.8054 - loss: 0.4267 - val_accuracy: 0.8230 - val_loss: 0.4012
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 274s 110ms/step - accuracy: 0.8176 - loss: 0.3987 - val_accuracy: 0.8108 - val_loss: 0.4307
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 323s 110ms/step - accuracy: 0.8339 - loss: 0.3759 - val_accuracy: 0.7736 - val_loss: 0.4972
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 275s 110ms/step - accuracy: 0.8452 - loss: 0.3544 - val_accuracy: 0.8142 - val_loss: 0.4375


### Se puede aumentar un poco mas aumentanto Epoch. Debido a horas y agotamiento con gatos y perros lo dejo hoy aqui.


In [ ]:
model.save("../models/dogs_vs_cats_final.h5")
print('Modelo guardado correctamente')

Modelo guardado correctamente
